In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

In [ ]:
df = pd.read_csv("../Data/Raw/triage_cleaned_working.csv")
diag_train = pd.read_csv("../Data/Raw/triage_train_diagnostics.csv")
clinical_slice_candidates = pd.read_csv("../Data/Raw/clinical_slice_candidates.csv")
error_slice_candidates = pd.read_csv("../Data/Raw/error_guided_slice_candidates.csv")

print(df.shape)
print(diag_train.shape)
print(clinical_slice_candidates.shape)
print(error_slice_candidates.shape)

Reconstruct the original train/val/test split

In [ ]:
from sklearn.model_selection import train_test_split

target_col = "y"

model_df = df.copy()

train_df, temp_df = train_test_split(
    model_df,
    test_size=0.30,
    stratify=model_df[target_col],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df[target_col],
    random_state=42
)

print(train_df.shape, val_df.shape, test_df.shape)

Train a baseline model on train, score validation

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

numeric_cols = [
    "age", "n_admissions",
    "heart_rate_final", "bp_systolic", "bp_diastolic" if "bp_diastolic" in train_df.columns else "bp_diastole",
    "respiratory", "temperature", "spo2",
    "vitals_missing_count"
]

# handle the naming safely
if "bp_diastolic" in numeric_cols and "bp_diastolic" not in train_df.columns:
    numeric_cols.remove("bp_diastolic")
    numeric_cols.append("bp_diastole")

categorical_cols = [
    "arrival_mode", "arrival_month", "arrival_day", "arrival_hour_bin"
]

binary_cols = [
    "abdominal_pain", "chest_pain", "back_pain", "unclassified",
    "ckd", "dm", "headache", "htn", "cough", "chd", "asthma",
    "bronchitis", "dizziness", "glaucoma", "hepatitis", "paralysis",
    "pneumonia", "sicklecell", "arm_pain", "breast_pain", "dental_pain",
    "diarrhea", "ear_pain", "eye_pain", "fever", "pelvic_pain",
    "arrival_mode_missing",
    "hr_missing", "pulse_rate_missing", "bp_systolic_missing",
    "bp_diastole_missing", "respiratory_missing", "temperature_missing", "spo2_missing",
    "heart_rate_disagreement_flag"
]

feature_cols = numeric_cols + categorical_cols + binary_cols

X_train = train_df[feature_cols].copy()
y_train = train_df["y"].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df["y"].copy()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

binary_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
        ("bin", binary_transformer, binary_cols),
    ]
)

val_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=3000, solver="lbfgs", random_state=42))
])

val_pipeline.fit(X_train, y_train)

val_df = val_df.copy()
val_df["pred_proba"] = val_pipeline.predict_proba(X_val)[:, 1]
val_df["pred_label_05"] = (val_df["pred_proba"] >= 0.5).astype(int)

val_df["logloss"] = -(
    val_df["y"] * np.log(np.clip(val_df["pred_proba"], 1e-8, 1-1e-8)) +
    (1 - val_df["y"]) * np.log(np.clip(1 - val_df["pred_proba"], 1e-8, 1-1e-8))
)

val_df["false_negative_05"] = (
    (val_df["y"] == 1) & (val_df["pred_label_05"] == 0)
).astype(int)

val_df["high_conf_contradiction"] = (
    ((val_df["pred_proba"] >= 0.80) & (val_df["y"] == 0)) |
    ((val_df["pred_proba"] <= 0.20) & (val_df["y"] == 1))
).astype(int)

val_loss_cutoff = val_df["logloss"].quantile(0.90)
val_df["high_loss_top10"] = (val_df["logloss"] >= val_loss_cutoff).astype(int)

Define slice metric helper for validation

In [ ]:
def slice_metrics_generic(data, mask, y_col="y", p_col="pred_proba", threshold=0.5, logloss_col="logloss"):
    sub = data.loc[mask].copy()
    n = len(sub)
    
    if n == 0:
        return None
    
    y = sub[y_col].values
    p = sub[p_col].values
    y_pred = (p >= threshold).astype(int)
    
    auroc = np.nan
    if len(np.unique(y)) > 1:
        auroc = roc_auc_score(y, p)
    
    auprc = average_precision_score(y, p)
    
    tn = fp = fn = tp = 0
    if len(np.unique(y)) > 1:
        tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    else:
        if y[0] == 1:
            tp = (y_pred == 1).sum()
            fn = (y_pred == 0).sum()
        else:
            tn = (y_pred == 0).sum()
            fp = (y_pred == 1).sum()
    
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    
    return {
        "n": n,
        "support_pct": 100 * n / len(data),
        "event_rate": y.mean(),
        "AUROC": auroc,
        "AUPRC": auprc,
        "Recall@0.5": recall,
        "Precision@0.5": precision,
        "avg_logloss": sub[logloss_col].mean(),
        "high_loss_rate": sub["high_loss_top10"].mean(),
        "high_conf_contradiction_rate": sub["high_conf_contradiction"].mean(),
        "fn_rate": sub["false_negative_05"].mean()
    }

Rebuild the slice definition registry

In [ ]:
all_slice_defs = {
    # --- Clinical / operational slices from Notebook 3 ---
    "chest_pain": lambda d: d["chest_pain"] == 1,
    "abdominal_pain": lambda d: d["abdominal_pain"] == 1,
    "back_pain": lambda d: d["back_pain"] == 1,
    "headache": lambda d: d["headache"] == 1,
    "cough": lambda d: d["cough"] == 1,
    "unclassified": lambda d: d["unclassified"] == 1,
    "ckd": lambda d: d["ckd"] == 1,
    "dm": lambda d: d["dm"] == 1,
    "htn": lambda d: d["htn"] == 1,
    "chd": lambda d: d["chd"] == 1,
    "pneumonia": lambda d: d["pneumonia"] == 1,
    "arrival_mode_ambulance": lambda d: d["arrival_mode"] == "ambulance",
    "arrival_mode_wheelchair": lambda d: d["arrival_mode"] == "Wheelchair",
    "arrival_mode_unknown": lambda d: d["arrival_mode"] == "Unknown",
    "vitals_missing_ge_3": lambda d: d["vitals_missing_count"] >= 3,
    "spo2_missing": lambda d: d["spo2_missing"] == 1,
    "hr_missing": lambda d: d["hr_missing"] == 1,
    "tachycardia": lambda d: d["tachycardia_flag"] == 1,
    "hypotension": lambda d: d["hypotension_flag"] == 1,
    "tachypnea": lambda d: d["tachypnea_flag"] == 1,
    "hypoxia": lambda d: d["hypoxia_flag"] == 1,
    "fever_temp": lambda d: d["fever_temp_flag"] == 1,
    "chest_pain_and_ambulance": lambda d: (d["chest_pain"] == 1) & (d["arrival_mode"] == "ambulance"),
    "abdominal_pain_and_hr_missing": lambda d: (d["abdominal_pain"] == 1) & (d["hr_missing"] == 1),
    "ckd_and_spo2_missing": lambda d: (d["ckd"] == 1) & (d["spo2_missing"] == 1),
    "pneumonia_and_tachypnea": lambda d: (d["pneumonia"] == 1) & (d["tachypnea_flag"] == 1),
    "unclassified_and_vitals_missing_ge_3": lambda d: (d["unclassified"] == 1) & (d["vitals_missing_count"] >= 3),

    # --- Error-guided slices from Notebook 4 (must match exactly) ---
    "err_vitals_missing_ge_3": lambda d: d["vitals_missing_count"] >= 3,
    "err_spo2_missing_and_unclassified": lambda d: (d["spo2_missing"] == 1) & (d["unclassified"] == 1),
    "err_hr_missing_and_abdominal_pain": lambda d: (d["hr_missing"] == 1) & (d["abdominal_pain"] == 1),
    "err_pneumonia_and_tachypnea": lambda d: (d["pneumonia"] == 1) & (d["tachypnea_flag"] == 1),
    "err_ckd_and_spo2_missing": lambda d: (d["ckd"] == 1) & (d["spo2_missing"] == 1),
    "err_chest_pain_and_tachycardia": lambda d: (d["chest_pain"] == 1) & (d["tachycardia_flag"] == 1),
    "err_unclassified_and_missingness": lambda d: (d["unclassified"] == 1) & (d["vitals_missing_count"] >= 2),
    "err_hypoxia_and_arrival_missing": lambda d: (d["hypoxia_flag"] == 1) & (d["arrival_mode_missing"] == 1),
}

Build a combined candidate list

In [ ]:
candidate_names = list(
    set(clinical_slice_candidates["slice_name"].tolist()) |
    set(error_slice_candidates["slice_name"].tolist())
)

candidate_names = [s for s in candidate_names if s in all_slice_defs]

print("Total candidate slices in registry:", len(candidate_names))
candidate_names[:20]

compute global train and validation references

In [ ]:
def slice_metrics_train(data, mask):
    return slice_metrics_generic(
        data.rename(columns={"oof_pred_proba": "pred_proba", "oof_logloss": "logloss"}),
        mask,
        y_col="y",
        p_col="pred_proba",
        threshold=0.5,
        logloss_col="logloss"
    )

global_train = slice_metrics_train(diag_train, diag_train.index == diag_train.index)
global_val = slice_metrics_generic(val_df, val_df.index == val_df.index)

print("Global train:", global_train)
print("Global val:", global_val)

In [ ]:
def add_physiology_flags(df):
    df = df.copy()

    df["tachycardia_flag"] = (
        df["heart_rate_final"].notna() & (df["heart_rate_final"] > 100)
    ).astype(int)

    df["hypotension_flag"] = (
        df["bp_systolic"].notna() & (df["bp_systolic"] < 90)
    ).astype(int)

    df["tachypnea_flag"] = (
        df["respiratory"].notna() & (df["respiratory"] > 20)
    ).astype(int)

    df["hypoxia_flag"] = (
        df["spo2"].notna() & (df["spo2"] < 95)
    ).astype(int)

    df["fever_temp_flag"] = (
        df["temperature"].notna() & (df["temperature"] >= 38.0)
    ).astype(int)

    df["hypothermia_flag"] = (
        df["temperature"].notna() & (df["temperature"] < 36.0)
    ).astype(int)

    return df

In [ ]:
diag_train = add_physiology_flags(diag_train)
val_df = add_physiology_flags(val_df)

In [ ]:
needed = [
    "tachycardia_flag", "hypotension_flag", "tachypnea_flag",
    "hypoxia_flag", "fever_temp_flag", "hypothermia_flag"
]

print("diag_train missing:", [c for c in needed if c not in diag_train.columns])
print("val_df missing:", [c for c in needed if c not in val_df.columns])

9 Define ASAP acceptance criteria 

Evaluate all candidate slices on train and validation

In [ ]:
rows = []

for slice_name in candidate_names:
    fn = all_slice_defs[slice_name]
    
    train_mask = fn(diag_train)
    val_mask = fn(val_df)
    
    train_m = slice_metrics_train(diag_train, train_mask)
    val_m = slice_metrics_generic(val_df, val_mask)
    
    if train_m is None or val_m is None:
        continue
    
    row = {
        "slice_name": slice_name,
        
        "train_n": train_m["n"],
        "train_support_pct": train_m["support_pct"],
        "train_event_rate": train_m["event_rate"],
        "train_AUPRC": train_m["AUPRC"],
        "train_Recall@0.5": train_m["Recall@0.5"],
        "train_avg_logloss": train_m["avg_logloss"],
        "train_high_loss_rate": train_m["high_loss_rate"],
        "train_high_conf_contradiction_rate": train_m["high_conf_contradiction_rate"],
        "train_fn_rate": train_m["fn_rate"],
        
        "val_n": val_m["n"],
        "val_support_pct": val_m["support_pct"],
        "val_event_rate": val_m["event_rate"],
        "val_AUPRC": val_m["AUPRC"],
        "val_Recall@0.5": val_m["Recall@0.5"],
        "val_avg_logloss": val_m["avg_logloss"],
        "val_high_loss_rate": val_m["high_loss_rate"],
        "val_high_conf_contradiction_rate": val_m["high_conf_contradiction_rate"],
        "val_fn_rate": val_m["fn_rate"],
    }
    
    rows.append(row)

audit_df = pd.DataFrame(rows)
audit_df.head()

Compute relative harm vs global references

In [ ]:
audit_df["train_logloss_lift"] = audit_df["train_avg_logloss"] - global_train["avg_logloss"]
audit_df["val_logloss_lift"] = audit_df["val_avg_logloss"] - global_val["avg_logloss"]

audit_df["train_recall_gap"] = audit_df["train_Recall@0.5"] - global_train["Recall@0.5"]
audit_df["val_recall_gap"] = audit_df["val_Recall@0.5"] - global_val["Recall@0.5"]

audit_df["train_fn_lift"] = audit_df["train_fn_rate"] - global_train["fn_rate"]
audit_df["val_fn_lift"] = audit_df["val_fn_rate"] - global_val["fn_rate"]

audit_df["train_hcc_lift"] = audit_df["train_high_conf_contradiction_rate"] - global_train["high_conf_contradiction_rate"]
audit_df["val_hcc_lift"] = audit_df["val_high_conf_contradiction_rate"] - global_val["high_conf_contradiction_rate"]

Encode ASAP support rules

In [ ]:
audit_df["pass_support"] = (
    (audit_df["train_support_pct"] >= 0.5) &
    (audit_df["val_support_pct"] >= 0.3)
)

Encode ASAP harm rules

In [ ]:
audit_df["pass_harm"] = (
    (audit_df["val_logloss_lift"] > 0.02) |
    (audit_df["val_recall_gap"] < -0.03) |
    (audit_df["val_fn_lift"] > 0.02) |
    (audit_df["val_hcc_lift"] > 0.01)
)

Encode ASAP persistence rules

In [ ]:
audit_df["persist_logloss"] = (
    (audit_df["train_logloss_lift"] > 0) &
    (audit_df["val_logloss_lift"] > 0)
)

audit_df["persist_recall"] = (
    (audit_df["train_recall_gap"] < 0) &
    (audit_df["val_recall_gap"] < 0)
)

audit_df["persist_fn"] = (
    (audit_df["train_fn_lift"] > 0) &
    (audit_df["val_fn_lift"] > 0)
)

audit_df["persist_hcc"] = (
    (audit_df["train_hcc_lift"] > 0) &
    (audit_df["val_hcc_lift"] > 0)
)

audit_df["pass_persistence"] = (
    audit_df["persist_logloss"] |
    audit_df["persist_recall"] |
    audit_df["persist_fn"] |
    audit_df["persist_hcc"]
)

Add manual plausibility / artifact review fields

In [ ]:
audit_df["manual_plausibility"] = "REVIEW"
audit_df["artifact_flag"] = "REVIEW"
audit_df["redundancy_flag"] = "REVIEW"
audit_df["slice_type_manual"] = "REVIEW"

Preliminary automatic acceptance before manual review

In [ ]:
audit_df["auto_accept_pre_manual"] = (
    audit_df["pass_support"] &
    audit_df["pass_harm"] &
    audit_df["pass_persistence"]
)

Rank accepted candidates

In [ ]:
audit_df["priority_score"] = (
    2.0 * audit_df["val_logloss_lift"].clip(lower=0).fillna(0) +
    2.0 * (-audit_df["val_recall_gap"].clip(upper=0)).fillna(0) +
    1.5 * audit_df["val_fn_lift"].clip(lower=0).fillna(0) +
    1.0 * audit_df["val_hcc_lift"].clip(lower=0).fillna(0)
)

audit_ranked = audit_df.sort_values(
    ["auto_accept_pre_manual", "priority_score"],
    ascending=[False, False]
)

audit_ranked.head(30)

Create the Approved Slice Registry template

In [ ]:
approved_slice_registry = audit_ranked.copy()

approved_slice_registry["final_accept"] = "REVIEW"
approved_slice_registry["accept_reason"] = ""
approved_slice_registry["reject_reason"] = ""

approved_slice_registry.head(20)

Save the audit registry

In [ ]:
approved_slice_registry.to_csv("approved_slice_registry_TEMPLATE.csv", index=False)
print("Saved approved_slice_registry_TEMPLATE.csv")

Manual Slice Review and Final Registry Construction

In [ ]:
registry = pd.read_csv("approved_slice_registry_TEMPLATE.csv")
registry.head(20)

Inspect the key col

In [ ]:
important_cols = [
    "slice_name",
    "train_support_pct", "val_support_pct",
    "train_logloss_lift", "val_logloss_lift",
    "train_recall_gap", "val_recall_gap",
    "train_fn_lift", "val_fn_lift",
    "train_hcc_lift", "val_hcc_lift",
    "pass_support", "pass_harm", "pass_persistence",
    "auto_accept_pre_manual",
    "manual_plausibility", "artifact_flag",
    "redundancy_flag", "slice_type_manual",
    "priority_score"
]

registry[important_cols].sort_values(
    ["auto_accept_pre_manual", "priority_score"],
    ascending=[False, False]
).head(30)

In [ ]:
registry.to_csv("approved_slice_registry_REVIEW.csv", index=False)
print("Saved approved_slice_registry_REVIEW.csv")

In [ ]:
registry = pd.read_csv("approved_slice_registry_REVIEWED.csv")
registry.head()

#Fixing 

In [ ]:
error_to_canonical = {
    "err_hr_missing_and_abdominal_pain": "abdominal_pain_and_hr_missing",
    "err_ckd_and_spo2_missing": "ckd_and_spo2_missing",
}

registry["canonical_slice_name"] = registry["slice_name"].replace(error_to_canonical)

registry["has_error_guided_evidence"] = registry["slice_name"].str.startswith("err_")

error_canonical_names = set(registry.loc[
    registry["has_error_guided_evidence"],
    "canonical_slice_name"
])

registry["has_error_guided_evidence"] = registry["canonical_slice_name"].isin(error_canonical_names)

registry["has_clinical_evidence"] = ~registry["slice_name"].str.startswith("err_")

registry["has_dual_evidence"] = (
    registry["has_clinical_evidence"] &
    registry["has_error_guided_evidence"]
)

registry[[
    "slice_name",
    "canonical_slice_name",
    "has_clinical_evidence",
    "has_error_guided_evidence",
    "has_dual_evidence"
]].head(20)

In [ ]:
#def decide_final_accept(row):
#    auto_ok = bool(row["auto_accept_pre_manual"])
#    
#    plausible_ok = str(row["manual_plausibility"]).strip().lower() in [
#        "plausible_clinical", "plausible_operational", "mixed"
#    ]
#    
#    artifact_ok = str(row["artifact_flag"]).strip().lower() not in [
#        "pure_artifact"
#    ]
#    
#    redundancy_val = str(row["redundancy_flag"]).strip().lower()
#    redundancy_ok = redundancy_val.startswith("unique") or redundancy_val == "unique"
#    
#    slice_type_ok = str(row["slice_type_manual"]).strip().lower() in [
#        "clinical_harm_slice",
#        "operational_harm_slice",
#        "mixed_harm_slice"
#    ]
#    
#    return "yes" if (auto_ok and plausible_ok and artifact_ok and redundancy_ok and slice_type_ok) else "no"
#
#registry["final_accept"] = registry.apply(decide_final_accept, axis=1)
#
#registry[[
#    "slice_name", "auto_accept_pre_manual", "manual_plausibility",
#    "artifact_flag", "redundancy_flag", "slice_type_manual", "final_accept"
#]].sort_values(["final_accept", "slice_name"], ascending=[False, True]).head(50)

In [ ]:
def decide_final_accept(row):
    auto_ok = bool(row["auto_accept_pre_manual"])

    plausible_ok = str(row["manual_plausibility"]).strip().lower() in [
        "plausible_clinical",
        "plausible_operational",
        "mixed"
    ]

    artifact = str(row["artifact_flag"]).strip().lower()

    artifact_ok = artifact not in [
        "pure_artifact",
        "mostly_missingness_artifact"
    ]

    redundancy_val = str(row["redundancy_flag"]).strip().lower()

    redundancy_ok = (
        redundancy_val.startswith("unique")
        or row.get("has_dual_evidence", False)
    )

    slice_type_ok = str(row["slice_type_manual"]).strip().lower() in [
        "clinical_harm_slice",
        "operational_harm_slice",
        "mixed_harm_slice"
    ]

    return "yes" if (
        auto_ok and plausible_ok and artifact_ok and redundancy_ok and slice_type_ok
    ) else "no"
    registry["final_accept"] = registry.apply(decide_final_accept, axis=1)
registry[[
    "slice_name", "auto_accept_pre_manual", "manual_plausibility",
    "artifact_flag", "redundancy_flag", "slice_type_manual", "final_accept"
]].sort_values(["final_accept", "slice_name"], ascending=[False, True]).head(50)

In [ ]:
registry["final_accept"].value_counts(dropna=False)

In [ ]:
final_accepted = registry[registry["final_accept"] == "yes"].copy()

final_accepted[[
    "slice_name",
    "slice_type_manual",
    "train_support_pct",
    "val_support_pct",
    "val_logloss_lift",
    "val_recall_gap",
    "val_fn_lift",
    "val_hcc_lift",
    "priority_score"
]].sort_values("priority_score", ascending=False)

In [ ]:
registry.to_csv("approved_slice_registry_FINAL.csv", index=False)
print("Saved approved_slice_registry_FINAL.csv")